# Performance Stats Analysis

This notebook rebuilds the test-set predictions using the best model from each fold, creates the 1623-row per-video analysis table, exports it to Excel, and generates summary visuals by participant, footwear, direction, and angle.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from CV.stats_analysis import (
    build_master_performance_table,
    export_analysis_excel,
    generate_sheet_visuals,
    make_standard_summary_tables,
    plot_error_rate_bars,
    plot_success_heatmap,
    plot_success_rate_bars,
)

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 200)

RUNS_ROOT = Path('CV/runs/ctr_gcn_kfold_hpo')
CV_SPLITS_DIR = Path('CV/data/cv_splits')
ANALYSIS_DIR = Path('CV/analysis_outputs')
FIGURES_DIR = ANALYSIS_DIR / 'figures'
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Set to 'cuda' to force GPU, 'cpu' to force CPU, or None to auto-detect.
DEVICE = None

In [ ]:
master_df, best_per_fold = build_master_performance_table(
    runs_root=RUNS_ROOT,
    cv_splits_dir=CV_SPLITS_DIR,
    device=DEVICE,
)
summary_tables = make_standard_summary_tables(master_df)

print(f'Per-video rows: {len(master_df)}')
display(best_per_fold[[
    'fold', 'run_name', 'hparam_key', 'best_val_bal_acc', 'best_val_epoch',
    'total_epochs', 'test_bal_acc', 'test_acc', 'test_tpr_fail', 'test_tnr_pass'
]])
display(master_df.head())

In [ ]:
chart_manifest = generate_sheet_visuals(
    master_df=master_df,
    best_per_fold=best_per_fold,
    summary_tables=summary_tables,
    output_dir=FIGURES_DIR,
)

excel_path = export_analysis_excel(
    master_df=master_df,
    best_per_fold=best_per_fold,
    summary_tables=summary_tables,
    output_path=ANALYSIS_DIR / 'performance_stats_analysis.xlsx',
)
print(f'Wrote Excel workbook: {excel_path}')
display(chart_manifest)

In [ ]:
display(summary_tables['by_participant'])
display(summary_tables['by_footwear'])
display(summary_tables['by_direction'])
display(summary_tables['by_angle'])

In [ ]:
fig, _ = plot_success_rate_bars(summary_tables['by_participant'], 'participant_id', 'Success Rate by Participant')
plt.show()

fig, _ = plot_error_rate_bars(summary_tables['by_participant'], 'participant_id', 'Error Rate by Participant')
plt.show()

fig, _ = plot_success_rate_bars(summary_tables['by_footwear'], 'footwear_id', 'Success Rate by Footwear')
plt.show()

fig, _ = plot_success_rate_bars(summary_tables['by_angle'], 'angle_deg', 'Success Rate by Angle', rotation=0)
plt.show()

fig, _ = plot_success_heatmap(master_df, 'footwear_id', 'angle_deg', 'Success Rate Heatmap: Footwear vs Angle')
plt.show()

fig, _ = plot_success_heatmap(master_df, 'participant_id', 'angle_deg', 'Success Rate Heatmap: Participant vs Angle')
plt.show()